# stockout — exploration

The five questions in [`docs/questions.md`](../docs/questions.md) were committed before
this notebook existed. Answer them here, then write the conclusions in
[`docs/results.md`](../docs/results.md) with the number attached.

**Two rules.**

1. Do not add a sixth question after seeing the data. Anything interesting that turns
   up goes in results.md under *what we did not ask*, clearly marked as unplanned.
2. Report the hypotheses that lose. A results file where all five won is evidence the
   questions were written after the charts.

The package holds anything with one correct answer; the judgement calls belong here,
visible next to their output ([ADR 0006](../docs/decisions/0006-analysis-in-notebooks.md)).

In [ ]:
import pandas as pd

from stockout.config import SAMPLE_PATH
from stockout.data.loaders import read_sales
from stockout.data.validate import calendar_gaps, null_profile, validate_sales
from stockout.evaluate.backtest import backtest, summarise
from stockout.features.build import build_features, feature_columns
from stockout.inventory.policy import critical_ratio
from stockout.inventory.simulate import frontier
from stockout.models import forecaster
from stockout.models.baselines import BASELINES, MovingAverage, NaiveLast, SeasonalNaive
from stockout.models.gbm import GbmForecaster, GbmQuantileForecaster
from stockout.plots import apply_style, save
from stockout.split.rolling import rolling_origin, split_frame

apply_style()

# Swap for data/raw/train.csv once `python -m stockout fetch` has run. Everything below
# works on the synthetic sample, but no finding may be reported from it.
DATA = SAMPLE_PATH

sales = read_sales(DATA)
validate_sales(sales)
sales.head()

In [ ]:
trading = sales[sales["open"] == 1]
print(f"rows        {len(sales):,}")
print(f"stores      {sales['store'].nunique():,}")
print(f"span        {sales['date'].min().date()} to {sales['date'].max().date()}")
print(f"trading     {len(trading) / len(sales):.1%} of rows")
display(null_profile(sales))
display(calendar_gaps(sales))

## Q1 — Does accuracy decay with horizon, and how fast?

**Hypothesis:** monotonic decay, steepest between 7 and 14 days.  
**Counts as a no:** WMAPE flat across horizons.

### Answer

*Unanswered.*

In [ ]:
# Scaffolded, not answered. Run it, read it, then write the conclusion above.
decay = pd.DataFrame(
    [
        {"horizon": h, **summarise(backtest(sales, SeasonalNaive, n_folds=3, horizon=h))}
        for h in (7, 14, 28, 42)
    ]
)
decay

## Q2 — Does seasonal-naive beat a GBM on low-volume stores?

**Hypothesis:** the GBM wins overall but loses on the bottom volume quartile.  
**Counts as a no:** the GBM wins uniformly across quartiles.

> The synthetic sample draws every store's base level from one uniform range, so its
> quartiles are close together by construction. This question needs Rossmann's 1115
> stores to mean anything.

### Answer

*Unanswered.*

In [ ]:
# Scaffolded, not answered. Fit once, then score each quartile separately — refitting
# per quartile would answer a different question (one model per segment, not one model
# serving segments of different size).
volume = trading.groupby("store")["sales"].mean().rename("mean_sales")
quartile = pd.qcut(volume, 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"], duplicates="drop")

fold = rolling_origin(sales["date"], n_folds=1, horizon=42)[-1]
train, test = split_frame(sales, fold)

predictions = {
    name: forecaster(name, horizon=42)().fit(train).predict(test)
    for name in ("seasonal_naive", "gbm")
}
scored = test.assign(quartile=test["store"].map(quartile), **predictions)
scored[scored["open"] == 1].head()

## Q3 — How much of the total error comes from a few days?

**Hypothesis:** heavily concentrated, clustered on holidays and promotion boundaries.  
**Counts as a no:** error spread evenly, so no special-case model is worth building.

### Answer

*Unanswered.*

In [ ]:
# Hint: refit a fold by hand with split.rolling.rolling_origin + split_frame, keep the
# per-row absolute errors, sort descending, and plot the cumulative share.

## Q4 — Does the promotion lift persist afterwards, or reverse?

**Hypothesis:** a dip. Promotions pull demand forward rather than creating it.  
**Counts as a no:** post-promotion sales at or above a matched baseline.

Note this one is invisible to WMAPE and only shows up in the inventory simulation:
stocking to a promotion forecast overstocks the following week.

### Answer

*Unanswered.*

In [ ]:
# Match on (store, day_of_week) so the weekday cycle cannot masquerade as a promo effect.

## Q5 — Does the newsvendor quantile beat stocking to the mean?

**Hypothesis:** the quantile policy reaches a higher fill rate at equal holding cost.  
**Counts as a no:** both policies land on the same frontier — the normal approximation
was good enough and the quantile models were not worth building.

This is the question the repository is named after, and a **no** is the most interesting
outcome available.

> Before trusting either curve, check the calibration cell below. On the synthetic
> sample every nominal quantile under-covers out of sample — a stated 0.9 delivers about
> 0.72 — which moves the cost-minimising level away from the derived critical ratio. A
> comparison of two policies where one is miscalibrated is a comparison of calibration.
> See [`docs/results.md`](../docs/results.md).

### Answer

*Unanswered.*

In [ ]:
# The target service level, derived from the cost pair rather than tuned.
target = critical_ratio()

quantile_model = GbmQuantileForecaster(horizon=42).fit(train)
quantiles = quantile_model.predict_quantiles(test)

# Inventory is held per store, so simulate one. Averaging a fill rate across a quiet
# shop and a busy one describes neither of them.
store = int(test["store"].iloc[0])
rows = test["store"] == store

curve = frontier(test.loc[rows, "sales"], quantiles.loc[rows])
print(f"target quantile {target:.2f} · crossing on {quantile_model.crossing_rate:.1%} of rows")
curve

In [ ]:
# Calibration, before the frontier is believed. A well-calibrated 0.9 is exceeded 10% of
# the time; one that covers 0.99 is not conservative, it is wrong, and it carries stock
# nobody needed. Trading days only — a shut store is predicted zero and covered trivially.
from stockout.evaluate.metrics import coverage

open_rows = test["open"] == 1
calibration = pd.DataFrame(
    [
        {
            "nominal": float(column),
            "covered": coverage(test.loc[open_rows, "sales"], quantiles.loc[open_rows, column]),
        }
        for column in quantiles.columns
    ]
)
calibration.assign(gap=lambda d: d["covered"] - d["nominal"])

---

## Before closing this notebook

- [ ] Every answer above written, with a number
- [ ] Conclusions copied into [`docs/results.md`](../docs/results.md)
- [ ] *What didn't work* filled in — a dead end costs a day either way, and writing it
      down is the only way that day buys anything
- [ ] Charts saved via `save(fig, "qN_name")` into `reports/` (gitignored, regenerated)
- [ ] Anything reused twice promoted into the package, where it picks up a test